# 🧑‍💻 TinyStories GPTNeo → llama2.c Export Notebook
This notebook shows how to inspect, convert, verify and export a Tiny GPTNeo model for minimal inference engines like llama2.c and esp32-llm.

In [1]:
!pip install transformers torch numpy

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import numpy as np
import struct

In [3]:
model_id = "roneneldan/TinyStories-Instruct-1M"
model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Model config:", model.config)
print("Total parameters:", sum(p.numel() for p in model.parameters()))
print("Vocab size:", tokenizer.vocab_size)

2025-06-22 14:47:00.986673: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-22 14:47:01.085768: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model config: GPTNeoConfig {
  "activation_function": "gelu_new",
  "architectures": [
    "GPTNeoForCausalLM"
  ],
  "attention_dropout": 0,
  "attention_layers": [
    "global",
    "local",
    "global",
    "local",
    "global",
    "local",
    "global",
    "local"
  ],
  "attention_types": [
    [
      [
        "global",
        "local"
      ],
      4
    ]
  ],
  "bos_token_id": 50256,
  "classifier_dropout": 0.1,
  "embed_dropout": 0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "hidden_size": 64,
  "initializer_range": 0.02,
  "intermediate_size": null,
  "layer_norm_epsilon": 1e-05,
  "max_position_embeddings": 2048,
  "model_type": "gpt_neo",
  "num_heads": 16,
  "num_layers": 8,
  "resid_dropout": 0,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "torch_dtype": "float32",
  "transformers_version": "4.52.4",
  "use_cache": true,
  "vocab_size":

In [4]:
layer0 = model.transformer.h[0]
print("Layer 0 attn dir:", dir(layer0.attn))
print("Layer 0 attn.attention dir:", dir(layer0.attn.attention))
print("Named parameters:", list(layer0.attn.attention.named_parameters()))

Layer 0 attn dir: ['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_is_hf_initialized', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_paramet

In [5]:
def safe_bias(proj):
    if proj.bias is None:
        print(f"Note: {proj} has no bias → using zeros")
        return np.zeros(proj.weight.shape[0], dtype=np.float32)
    else:
        return proj.bias.data.cpu().numpy()

# Test
q_proj = layer0.attn.attention.q_proj
print("Q weight shape:", q_proj.weight.shape)
print("Q bias shape:", "None" if q_proj.bias is None else q_proj.bias.shape)
print("Safe bias:", safe_bias(q_proj))

Q weight shape: torch.Size([64, 64])
Q bias shape: None
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Safe bias: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [6]:
vocab_size = model.config.vocab_size
hidden_size = model.config.hidden_size
num_layers = model.config.num_layers
num_heads = model.config.num_heads
layer_norm_eps = model.config.layer_norm_epsilon
max_seq_len = model.config.max_position_embeddings

with open("tinystories_gptneo.bin", "wb") as f:
    # Magic header
    f.write(b'GPNO')
    f.write(struct.pack("i", vocab_size))
    f.write(struct.pack("i", hidden_size))
    f.write(struct.pack("i", num_layers))
    f.write(struct.pack("i", num_heads))
    f.write(struct.pack("f", layer_norm_eps))
    f.write(struct.pack("i", max_seq_len))

    # Embeddings
    f.write(model.transformer.wte.weight.data.cpu().numpy().tobytes())
    f.write(model.transformer.wpe.weight.data.cpu().numpy().tobytes())

    for i in range(num_layers):
        layer = model.transformer.h[i]
        print(f"Layer {i}")

        f.write(layer.ln_1.weight.data.cpu().numpy().tobytes())
        f.write(layer.ln_1.bias.data.cpu().numpy().tobytes())

        attn = layer.attn.attention
        q_proj, k_proj, v_proj, out_proj = attn.q_proj, attn.k_proj, attn.v_proj, attn.out_proj

        f.write(q_proj.weight.data.cpu().numpy().tobytes())
        f.write(k_proj.weight.data.cpu().numpy().tobytes())
        f.write(v_proj.weight.data.cpu().numpy().tobytes())
        f.write(out_proj.weight.data.cpu().numpy().tobytes())

        f.write(safe_bias(q_proj).tobytes())
        f.write(safe_bias(k_proj).tobytes())
        f.write(safe_bias(v_proj).tobytes())
        f.write(safe_bias(out_proj).tobytes())

        f.write(layer.ln_2.weight.data.cpu().numpy().tobytes())
        f.write(layer.ln_2.bias.data.cpu().numpy().tobytes())

        f.write(layer.mlp.c_fc.weight.data.cpu().numpy().tobytes())
        f.write(layer.mlp.c_fc.bias.data.cpu().numpy().tobytes())
        f.write(layer.mlp.c_proj.weight.data.cpu().numpy().tobytes())
        f.write(layer.mlp.c_proj.bias.data.cpu().numpy().tobytes())

    f.write(model.transformer.ln_f.weight.data.cpu().numpy().tobytes())
    f.write(model.transformer.ln_f.bias.data.cpu().numpy().tobytes())
    f.write(model.lm_head.weight.data.cpu().numpy().tobytes())

    print("✅ Export done.")
!ls -lh tinystories_gptneo.bin

Layer 0
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Layer 1
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Layer 2
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Layer 3
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Note: Linear(in_features=64, out_features=64, bias=False) has no bias → using zeros
Note: Linear(in_features=64, out_features=64

In [8]:
with open("tinystories_gptneo.bin", "rb") as f:
    magic = f.read(4)
    assert magic == b'GPNO'
    vocab_size = struct.unpack("i", f.read(4))[0]
    hidden_size = struct.unpack("i", f.read(4))[0]
    num_layers = struct.unpack("i", f.read(4))[0]
    num_heads = struct.unpack("i", f.read(4))[0]
    layer_norm_eps = struct.unpack("f", f.read(4))[0]
    max_seq_len = struct.unpack("i", f.read(4))[0]

    print("Header:", vocab_size, hidden_size, num_layers, num_heads, layer_norm_eps, max_seq_len)

    embed = np.frombuffer(f.read(vocab_size * hidden_size * 4), dtype=np.float32)
    print("Embeddings shape:", embed.shape)
    pos_embed = np.frombuffer(f.read(max_seq_len * hidden_size * 4), dtype=np.float32)
    print("Positional embeddings shape:", pos_embed.shape)
    print("LM head shape:", model.lm_head.weight.shape)
    print("First layer MLP c_fc weight:", model.transformer.h[0].mlp.c_fc.weight.shape)
    print("First layer MLP c_proj weight:", model.transformer.h[0].mlp.c_proj.weight.shape)

Header: 50257 64 8 16 9.999999747378752e-06 2048
Embeddings shape: (3216448,)
Positional embeddings shape: (131072,)
LM head shape: torch.Size([50257, 64])
First layer MLP c_fc weight: torch.Size([256, 64])
First layer MLP c_proj weight: torch.Size([64, 256])


✅ **All done!**

Let's verify:

| Field                                              | Value               | Meaning                                            | ✅ OK? |
| -------------------------------------------------- | ------------------- | -------------------------------------------------- | ----- |
| `Header: 50257 64 8 16 9.999999747378752e-06 2048` |                     |                                                    |       |
| **50257**                                          | vocab size          | TinyStories uses the GPT tokenizer — standard size | ✅     |
| **64**                                             | hidden size         | small for edge                                     | ✅     |
| **8**                                              | num layers          | good for low RAM                                   | ✅     |
| **16**                                             | num heads           | so each head = hidden\_size / num\_heads = 4       | ✅     |
| **1e-5**                                           | layer norm epsilon  | matches config                                     | ✅     |
| **2048**                                           | max sequence length | classic GPT2 limit                                 | ✅     |


Vocabulary: 50257
Embeddings: vocab_size × hidden_size = 50257 × 64 = 3,216,448
Positional embeddings: max_seq_len × hidden_size = 2048 × 64 = = 131,072

👉 Our binary now has a valid header + embeddings, just like llama2.c expects:

Check the binary size, deploy it to your llama2.c or esp32-llm runtime, and test your new tiny LLM on your device!

We need to patch the constants in our run.c to match the new model:

```C
#define VOCAB_SIZE 50257
#define DIM 64
#define N_LAYERS 8
#define N_HEADS 16
#define MAX_SEQ_LEN 2048
```

## File Size

- 1️⃣ Header: a few bytes (negligible)
- 2️⃣ Token embeddings: vocab_size × hidden_size × 4 bytes
- 3️⃣ Positional embeddings: max_seq_len × hidden_size × 4 bytes
- 4️⃣ For each layer:
   - Norm1: 2 × hidden_size × 4 bytes
   - Q,K,V,Out weights: 4 × hidden_size² × 4 bytes
   - Q,K,V,Out biases: 4 × hidden_size × 4 bytes
   - Norm2: 2 × hidden_size × 4 bytes
   - MLP: c_fc weight + bias + c_proj weight + bias
          = hidden_size × (4 × hidden_size) + hidden_size × (4 × hidden_size) + biases
- 5️⃣ Final norm: 2 × hidden_size × 4 bytes
- 6️⃣ LM head: vocab_size × hidden_size × 4 bytes

| Component        | Approx. size                         |
| ---------------- | ------------------------------------ |
| Token embeddings | 50,257 × 64 × 4 = \~12.8 MB          |
| Pos embeddings   | 2048 × 64 × 4 = \~0.5 MB             |
| LM head          | same as token embeddings = \~12.8 MB |
| Layers           | this is the big one:                 |
| each layer:      |                                      |

- 2 Norm1 = 512 B
- 4 weight matrices QKVOut: 4 × 64 × 64 × 4 = 64 KB
- 4 biases QKVOut: 4 × 64 × 4 = 1 KB
- 2 Norm2 = 512 B

- MLP:
   - c_fc weight: 64 × (4×64) × 4 = 64 KB
   - c_fc bias: 256 × 4 = 1 KB
   - c_proj weight: 256 × 64 × 4 = 64 KB
   - c_proj bias: 64 × 4 = 256 B

-> Total per layer: ~200–300 KB

For 8 layers: ~1.5–2 MB

### Total Expected Model Size
12.8M (embed) + 0.5M (pos) + 12.8M (lm head) + ~2M (layers) ≈ 28 MB